# Impact Split Explainer

This notebook tells the development story of `impact-split` as a new approach for additive KPIs.

Flow of this notebook:

1. Why this problem needs a different lens
2. The working mathematical frame used in this notebook
3. A toy scenario to make the idea concrete
4. What to inspect in the generated tree and segments
5. How parameter choices change what we see

If you only need a high-level summary, use the README. This notebook is the deeper walkthrough.

## 1) Problem Framing

For additive KPIs, a useful question is often:

> Which segments contribute the most to total outcome (positive or negative)?

A simple intuition:

- Segment A: 2 users x -500 average = -1,000 total
- Segment B: 10,000 users x -40 average = -400,000 total

This notebook uses that framing to motivate the approach explored below.

## 2) Working Mathematical Frame

The definitions below describe the frame used in this notebook run.

### A. Local Sieve (`delta`)

At each node:

$$
V_{node} = \sum_{j=1}^{N_{node}} |y_j|
$$

$$
\delta = V_{node} \times \text{delta\_pct}
$$

For each category in feature $X_i$:

$$
S_{cat} = \sum_{y \in cat} y
$$

Routing rule in this frame:

- Positive if $S_{cat} > \delta$
- Negative if $S_{cat} < -\delta$
- Neutral otherwise

### B. Split Metric

$$
Gain(X_i) = \frac{|S_P|}{k_P} + \frac{|S_N|}{k_N}
$$

Where $S_P, S_N$ are outer-branch sums and $k_P, k_N$ are the number of categories assigned to each outer branch.

### C. Global Stopping Rule

Let:

$$
V_{global\_P} = \sum_{y_i > 0} y_i,
\quad
V_{global\_N} = \sum_{y_i < 0} |y_i|
$$

At each node:

$$
S_{node\_P} = \sum_{y_i \in node, y_i > 0} y_i,
\quad
S_{node\_N} = \sum_{y_i \in node, y_i < 0} |y_i|
$$

Continue splitting only if either ratio is above `min_global_impact_pct`:

- $S_{node\_P} / V_{global\_P}$
- $S_{node\_N} / V_{global\_N}$

Parameter mapping used later in code:

- `delta_pct` controls the local sieve width (`delta`)
- `min_global_impact_pct` controls the global stop threshold

In [ ]:
import numpy as np
import pandas as pd

from impact_split import ImpactSplitter

In [ ]:
# 3) Build a toy scenario for illustration (not a claim about real data)
rng = np.random.default_rng(42)
n = 5000

regions = np.array(["NCR", "Luzon", "Visayas", "Mindanao"])
channels = np.array(["Direct", "Partner", "Online"])
products = np.array(["A", "B", "C"])

X = pd.DataFrame(
    {
        "region": rng.choice(regions, size=n, p=[0.35, 0.3, 0.2, 0.15]),
        "channel": rng.choice(channels, size=n, p=[0.25, 0.35, 0.4]),
        "product": rng.choice(products, size=n, p=[0.4, 0.35, 0.25]),
    }
)

base = rng.normal(loc=0, scale=35, size=n)

# Inject simple, visible patterns so we can inspect how the tree responds.
impact = (
    np.where((X["region"] == "NCR") & (X["channel"] == "Direct"), 120, 0)
    + np.where((X["region"] == "Mindanao") & (X["product"].isin(["A", "B"])), -95, 0)
    + np.where((X["channel"] == "Online") & (X["product"] == "C"), 35, 0)
)

y = pd.Series(base + impact, name="profit_loss")

X.head(), y.head()

In [ ]:
# 4) Fit one baseline configuration for exploration
model = ImpactSplitter(
    delta_pct=0.05,
    min_global_impact_pct=0.01,
    max_depth=4,
)

model.fit(X, y)

In [ ]:
# Inspect the split structure produced by the baseline run.
model.plot_tree(figsize=(16, 10))

In [ ]:
# Inspect top segments and compare them with the patterns injected above.
segments = model.get_impact_segments()
segments.head(15)

## 5) Parameter Exploration

This section is intentionally exploratory.

Rather than assume one "correct" setting, compare how the output changes when we vary:

- `delta_pct` (local sieve width)
- `min_global_impact_pct` (global materiality stop)

Suggested reading questions:

- Which segments stay stable across runs?
- Which segments appear only under more aggressive settings?
- How does tree complexity change?

In [ ]:
# Run a small comparison grid and review differences in top segments.
for delta_pct, min_global_impact_pct in [(0.03, 0.01), (0.05, 0.01), (0.08, 0.02)]:
    m = ImpactSplitter(
        delta_pct=delta_pct,
        min_global_impact_pct=min_global_impact_pct,
        max_depth=4,
    )
    m.fit(X, y)
    top = m.get_impact_segments().head(3)
    print(f"delta_pct={delta_pct}, min_global_impact_pct={min_global_impact_pct}")
    print(top[["path", "total_sum"]] if {"path", "total_sum"}.issubset(top.columns) else top)
    print("-" * 80)